In [ ]:
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer


In [ ]:
tokenizer = Tokenizer(models.BPE())

In [ ]:
tokenizer

In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

In [ ]:
tokenizer.pre_tokenizer

In [ ]:
special_tokens_num=36
special_tokens_list = [
    "<|endoftext|>", "<|im_start|>", "<|im_end|>",
    "<|object_ref_start|>", "<|object_ref_end|>", "<|box_start|>", "<|box_end|>", "<|quad_start|>", "<|quad_end|>",
    "<|vision_start|>", "<|vision_end|>", "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    "<|audio_start|>", "<|audio_end|>", "<|audio_pad|>", "<tts_pad>", "<tts_text_bos>", "<tts_text_eod>", "<tts_text_bos_single>"
]

# 这一组 token 更多偏向“训练数据协议”：
# 比如工具调用、工具返回、显式思维链边界等。
# 这些 token 需要出现在词表里，但不一定都要在 Transformers 侧被当作 special token 处理。
additional_tokens_list = [
    "<tool_call>", "</tool_call>",
    "<tool_response>", "</tool_response>",
    "<think>", "</think>"
]

# 为了让总特殊 token 数固定到 `SPECIAL_TOKENS_NUM`，
# 把剩余槽位用 buffer token 预留出来，方便未来扩展。
num_buffer = special_tokens_num - len(special_tokens_list + additional_tokens_list)
buffer_tokens = [f"<|buffer{i}|>" for i in range(1, num_buffer + 1)]
all_special_tokens = special_tokens_list + additional_tokens_list + buffer_tokens


In [ ]:
all_special_tokens

In [ ]:
trainer = trainers.BpeTrainer(
    vocab_size=6400,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    specical_tokens=all_special_tokens,
)

In [ ]:
import json, os

In [ ]:

def get_texts(data_path=r'D:\Proj\minimind\data\sft_t2t_mini.jsonl'):
    """
    从 jsonl 数据中抽取可用于 tokenizer 训练的纯文本。

    数据格式预期类似：
    {
        "conversations": [
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]
    }

    这里把一条样本里的多轮 `content` 用换行拼起来，原因是：
    1. tokenizer 训练只需要“文本流”，不需要标签；
    2. 保留换行可以让 BPE 看到更接近真实对话的边界形态；
    3. 对教学数据来说，这样实现最直接，也足够稳定。
    """
    with open(data_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            # 这里只取前 10000 行，避免教学演示时训练过慢。
            # 真正大规模训练 tokenizer 时，通常会喂入更大、更干净、分布更全面的文本集合。
            if i >= 10000:
                break
            try:
                data = json.loads(line)
                # conversations 里每个 turn 只抽取 content 字段。
                # role 暂时不参与拼接，因为 role 控制通常由 chat template 和特殊 token 负责。
                contents = [item.get('content') for item in data.get('conversations', []) if item.get('content')]
                if contents:
                    yield "\n".join(contents)
            except json.JSONDecodeError:
                # 教学数据里如果偶尔混入坏行，这里选择跳过而不是让整个训练中断。
                continue

In [ ]:
texts = get_texts()

In [ ]:
next(texts)